In [1]:
import Util.ExploratoryDataAnalysis
import Util.ModelDevelopment
import Util.EvaluationEngine
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier


In [2]:
# Load game data
data_dir = Path("data")

dfs = [
    Util.ExploratoryDataAnalysis.load_battles_file_to_df(file)
    for file in data_dir.glob("*.json")
]

df = pd.concat(dfs, ignore_index=True)

df = pd.get_dummies(df, columns=['result'])

print(df.shape)
df.head()

(495310, 28)


,battleTime,gameModeId,playerTag,opponentTag,playerCrowns,opponentCrowns,crown_diff,p_card_1,o_card_1,p_card_2,...,o_card_6,p_card_7,o_card_7,p_card_8,o_card_8,p_card_9,o_card_9,result_draw,result_loss,result_win
0,20260303T044817.000Z,72000006,#9CVVPULRY,#9RRRV9PLP,1,0,1,26000007,26000017,26000011,...,26000004,26000021,26000043,28000007,28000011,159000000,159000000,0,0,1
1,20260303T040122.000Z,72000006,#9CVVPULRY,#2200YRUGGU,3,0,3,26000007,27000013,26000011,...,28000026,26000021,28000013,28000007,26000101,159000000,159000000,0,0,1
2,20260303T035814.000Z,72000006,#9CVVPULRY,#VYGPRCQLJ,0,1,-1,26000007,26000011,26000011,...,28000004,26000021,26000021,28000007,28000001,159000000,159000000,0,1,0
3,20260303T034754.000Z,72000006,#9CVVPULRY,#2LRLG2C28,1,0,1,26000007,26000064,26000011,...,26000048,26000021,28000003,28000007,26000042,159000000,159000000,0,0,1
4,20260303T034441.000Z,72000006,#9CVVPULRY,#8UJQQCQR,1,0,1,26000007,26000007,26000011,...,28000015,26000021,26000005,28000007,26000044,159000000,159000000,0,0,1


In [3]:
# Get the list of all the cards in the game from Clash Royale API
cards_df = Util.ExploratoryDataAnalysis.fetch_all_cards()

display(cards_df)

,name,id,maxLevel,maxEvolutionLevel,elixirCost,rarity,is_support
0,Knight,26000000,16,3.0,3.0,common,False
1,Archers,26000001,16,1.0,3.0,common,False
2,Goblins,26000002,16,2.0,2.0,common,False
3,Giant,26000003,14,2.0,5.0,rare,False
4,P.E.K.K.A,26000004,11,1.0,7.0,epic,False
...,...,...,...,...,...,...,...
120,Vines,28000026,11,NaN,3.0,epic,False
121,Tower Princess,159000000,16,NaN,NaN,common,True
122,Cannoneer,159000001,11,NaN,NaN,epic,True
123,Dagger Duchess,159000002,8,NaN,NaN,legendary,True


In [4]:
# Data Preprocessing
# Engineer metadata features and build presence matrix
X, y, artifacts = Util.ModelDevelopment.build_features_and_target(df, cards_df, n_cards_per_side=9, return_artifacts=True)

print(X.shape)
X.head()

(495310, 267)


,gameModeId,hour,day_of_week,p_has_26000000,p_has_26000001,p_has_26000002,p_has_26000003,p_has_26000004,p_has_26000005,p_has_26000006,...,p_building_count,o_avg_elixir,o_total_elixir,o_troop_count,o_spell_count,o_building_count,avg_elixir_diff,spell_diff,building_diff,troop_diff
0,72000006,4,1,0,0,0,0,0,0,0,...,1,4.714286,33.0,5,3,0,-1.339286,-1,1,0
1,72000006,4,1,0,0,0,0,0,0,0,...,1,4.000000,32.0,4,2,2,-0.625000,0,-1,1
2,72000006,3,1,0,0,0,0,0,0,0,...,1,3.125000,25.0,6,2,0,0.250000,0,1,-1
3,72000006,3,1,0,0,0,0,0,0,0,...,1,4.750000,38.0,7,1,0,-1.375000,1,1,-2
4,72000006,3,1,0,0,0,0,0,0,0,...,1,4.000000,32.0,7,1,0,-0.625000,1,1,-2


In [5]:
# Using the parameters from the model evaluation
# Best params: {'min_samples_split': 15, 'min_samples_leaf': 2, 'max_samples': 0.8, 'max_features': 'sqrt', 'max_depth': 20, 'bootstrap': True}

rf_model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=20,
    min_samples_split=15,
    min_samples_leaf=2,
    max_features="sqrt",
    max_samples=0.8,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X, y)

RandomForestClassifier(max_depth=20, max_samples=0.8, min_samples_leaf=2,
                       min_samples_split=15, n_estimators=1000, n_jobs=-1,
                       random_state=42)

In [6]:
FEATURE_COLUMNS = list(X.columns)

HogEQ = [
    "Hog Rider", "Earthquake", "The Log", "Firecracker",
    "Cannon", "Knight", "Ice Spirit", "Skeletons",
    "Tower Princess"  # tower troop
]

MinerRocket = [
    "Miner", "Rocket", "The Log", "Dark Prince",
    "Bomb Tower", "Royal Delivery", "Ice Spirit", "Skeletons",
    "Tower Princess"
]

win_pct, X_row = Util.EvaluationEngine.predict_matchup_win_pct(
    rf_model,
    HogEQ, MinerRocket,
    cards_df=cards_df,
    feature_columns=FEATURE_COLUMNS,
    gameModeId=72000006,
    hour=12,
    day_of_week=2
)

print(f"Deck A win probability vs Deck B: {win_pct:.1f}%")

Deck A win probability vs Deck B: 52.2%


In [9]:
log_bait = [
    "Goblin Barrel",
    "Princess",
    "Knight",
    "Goblin Gang",
    "Tesla",
    "The Log",
    "Rocket",
    "Ice Spirit",
    "Tower Princess"
]

xbow_cycle = [
    "X-Bow",
    "Tesla",
    "Knight",
    "Archers",
    "Skeletons",
    "Ice Spirit",
    "Fireball",
    "The Log",
    "Tower Princess"
]

golem = [
    "Golem",
    "Night Witch",
    "Baby Dragon",
    "Lightning",
    "Tornado",
    "Lumberjack",
    "Barbarian Barrel",
    "Bomber",
    "Tower Princess"
]

giant = [
    "Giant",
    "Dark Prince",
    "Prince",
    "Mega Minion",
    "Minions",
    "Zap",
    "Fireball",
    "Bomber",
    "Tower Princess"
]

royal_giant = [
    "Royal Giant",
    "Fisherman",
    "Hunter",
    "Royal Ghost",
    "Lightning",
    "The Log",
    "Skeletons",
    "Electro Spirit",
    "Tower Princess"
]

decks = {
    "Log Bait": log_bait,
    "Xbow": xbow_cycle,
    "Golem": golem,
    "Giant": giant,
    "Royal Giant": royal_giant
}

for a_name, deckA in decks.items():
    for b_name, deckB in decks.items():
        if a_name == b_name:
            continue

        # A vs B
        p_ab, _ = Util.EvaluationEngine.predict_matchup_win_pct(
            rf_model,
            deckA,
            deckB,
            cards_df=cards_df,
            feature_columns=FEATURE_COLUMNS
        )

        # B vs A
        p_ba, _ = Util.EvaluationEngine.predict_matchup_win_pct(
            rf_model,
            deckB,
            deckA,
            cards_df=cards_df,
            feature_columns=FEATURE_COLUMNS
        )

        advantage = p_ab - p_ba

        print(f"\n{a_name} vs {b_name}")
        print(f"A beats B: {p_ab:.1f}%")
        print(f"B beats A: {p_ba:.1f}%")
        print(f"Advantage: {advantage:+.1f}%")


Log Bait vs Xbow
A beats B: 50.0%
B beats A: 55.9%
Advantage: -5.9%

Log Bait vs Golem
A beats B: 57.5%
B beats A: 54.9%
Advantage: +2.6%

Log Bait vs Giant
A beats B: 53.4%
B beats A: 53.2%
Advantage: +0.1%

Log Bait vs Royal Giant
A beats B: 44.0%
B beats A: 59.3%
Advantage: -15.3%

Xbow vs Log Bait
A beats B: 55.9%
B beats A: 50.0%
Advantage: +5.9%

Xbow vs Golem
A beats B: 55.7%
B beats A: 56.0%
Advantage: -0.3%

Xbow vs Giant
A beats B: 53.2%
B beats A: 54.0%
Advantage: -0.8%

Xbow vs Royal Giant
A beats B: 44.6%
B beats A: 59.3%
Advantage: -14.7%

Golem vs Log Bait
A beats B: 54.9%
B beats A: 57.5%
Advantage: -2.6%

Golem vs Xbow
A beats B: 56.0%
B beats A: 55.7%
Advantage: +0.3%

Golem vs Giant
A beats B: 53.1%
B beats A: 60.0%
Advantage: -7.0%

Golem vs Royal Giant
A beats B: 53.3%
B beats A: 60.0%
Advantage: -6.7%

Giant vs Log Bait
A beats B: 53.2%
B beats A: 53.4%
Advantage: -0.1%

Giant vs Xbow
A beats B: 54.0%
B beats A: 53.2%
Advantage: +0.8%

Giant vs Golem
A beats B: 6

In [8]:
result = Util.EvaluationEngine.evaluate_matchup(
    rf_model,
    log_bait,
    golem,
    cards_df=cards_df,
    feature_columns=FEATURE_COLUMNS
)

print(result)

{'A_beats_B': 57.544494434357276, 'B_beats_A': 54.94646432333522, 'advantage': 2.598030111022055}
